In [1]:
import pandas as pd

periods = ["2010/11","2011/12","2012/13","2013/14","2014/15","2015/16","2016/17","2017/18",
           "2018/19","2019/20","2020/21","2021/22","2022/23","2023/24","2024/25"]

england = [84.2,86.0,87.7,88.3,88.6,88.2,87.6,87.2,86.4,86.8,86.6,85.7,84.5,83.9,83.7]

areas_data = {
    "Birmingham": {
        "value": [87.5,82.9,81.2,84.3,85.0,81.4,82.9,81.6,82.3,81.4,78.0,76.7,75.1,74.8,75.5],
        "lci":   [86.9,82.3,80.6,83.7,84.4,80.8,82.3,81.1,81.7,80.8,77.3,76.1,74.4,74.1,74.9],
        "uci":   [88.0,83.5,81.8,84.8,85.5,82.0,83.4,82.2,82.9,81.9,78.6,77.4,75.7,75.4,76.2],
        "count": [11691,12772,12796,13537,13760,13734,14582,14687,14678,14229,13566,13511,12798,12610,12355],
    },
    # Add Surrey here: change the Area dropdown to Surrey and copy the same four columns
    # "Surrey": {"value": [...], "lci": [...], "uci": [...], "count": [...]},
}

rows = []
for area, v in areas_data.items():
    assert all(len(v[k]) == 15 for k in v), f"{area}: each list needs 15 values"
    for i, p in enumerate(periods):
        rows.append({"area": area, "period": p, "coverage": v["value"][i],
                     "lci": v["lci"][i], "uci": v["uci"][i], "vaccinated": v["count"][i]})
for i, p in enumerate(periods):
    rows.append({"area": "England", "period": p, "coverage": england[i],
                 "lci": None, "uci": None, "vaccinated": None})

d = pd.DataFrame(rows)
d["start_year"] = d["period"].str[:4].astype(int)
eng_map = dict(zip(periods, england))
d["england_value"] = d["period"].map(eng_map)

d["gap_to_95"] = (95 - d["coverage"]).round(1)
d["gap_to_england"] = (d["england_value"] - d["coverage"]).round(1)
d["sig_below_95"] = d["uci"] < 95
d["sig_below_england"] = (d["uci"] < d["england_value"]) & (d["area"] != "England")
d = d.sort_values(["area", "start_year"])
d["change_vs_prev_year"] = d.groupby("area")["coverage"].diff().round(1)

d.to_csv("mmr2_clean.csv", index=False)

latest = d[d["period"] == "2024/25"]
print(latest[["area","coverage","lci","uci","gap_to_95","gap_to_england","sig_below_95","sig_below_england"]])
first = d[d["period"] == "2010/11"].set_index("area")["coverage"]
last = d[d["period"] == "2024/25"].set_index("area")["coverage"]
print("Change 2010/11 to 2024/25 (percentage points):\n", (last - first).round(1))

          area  coverage   lci   uci  gap_to_95  gap_to_england  sig_below_95  \
14  Birmingham      75.5  74.9  76.2       19.5             8.2          True   
29     England      83.7   NaN   NaN       11.3             0.0         False   

    sig_below_england  
14               True  
29              False  
Change 2010/11 to 2024/25 (percentage points):
 area
Birmingham   -12.0
England       -0.5
Name: coverage, dtype: float64


In [2]:
import pandas as pd

periods = ["2010/11","2011/12","2012/13","2013/14","2014/15","2015/16","2016/17","2017/18",
           "2018/19","2019/20","2020/21","2021/22","2022/23","2023/24","2024/25"]

england = [84.2,86.0,87.7,88.3,88.6,88.2,87.6,87.2,86.4,86.8,86.6,85.7,84.5,83.9,83.7]

areas_data = {
    "Birmingham": {
        "value": [87.5,82.9,81.2,84.3,85.0,81.4,82.9,81.6,82.3,81.4,78.0,76.7,75.1,74.8,75.5],
        "lci":   [86.9,82.3,80.6,83.7,84.4,80.8,82.3,81.1,81.7,80.8,77.3,76.1,74.4,74.1,74.9],
        "uci":   [88.0,83.5,81.8,84.8,85.5,82.0,83.4,82.2,82.9,81.9,78.6,77.4,75.7,75.4,76.2],
        "count": [11691,12772,12796,13537,13760,13734,14582,14687,14678,14229,13566,13511,12798,12610,12355],
    },
    "Surrey": {
        "value": [70.1,75.3,76.4,73.5,74.1,74.8,77.6,81.7,79.0,83.3,84.3,83.9,83.5,83.7,87.0],
        "lci":   [69.4,74.6,75.7,72.9,73.5,74.1,77.0,81.1,78.3,82.7,83.7,83.3,82.8,83.1,86.4],
        "uci":   [70.9,76.0,77.0,74.2,74.8,75.4,78.3,82.3,79.6,83.9,84.9,84.5,84.1,84.3,87.6],
        "count": [10048,11525,11995,11951,12147,12295,12716,13118,13206,12031,12521,12423,12194,12476,12162],
    },
}

rows = []
for area, v in areas_data.items():
    assert all(len(v[k]) == 15 for k in v), f"{area}: each list needs 15 values"
    for i, p in enumerate(periods):
        rows.append({"area": area, "period": p, "coverage": v["value"][i],
                     "lci": v["lci"][i], "uci": v["uci"][i], "vaccinated": v["count"][i]})
for i, p in enumerate(periods):
    rows.append({"area": "England", "period": p, "coverage": england[i],
                 "lci": None, "uci": None, "vaccinated": None})

d = pd.DataFrame(rows)
d["start_year"] = d["period"].str[:4].astype(int)
d["england_value"] = d["period"].map(dict(zip(periods, england)))

d["gap_to_95"] = (95 - d["coverage"]).round(1)
d["gap_to_england"] = (d["england_value"] - d["coverage"]).round(1)
d["sig_below_95"] = d["uci"] < 95
d["sig_below_england"] = (d["uci"] < d["england_value"]) & (d["area"] != "England")
d = d.sort_values(["area", "start_year"])
d["change_vs_prev_year"] = d.groupby("area")["coverage"].diff().round(1)

d.to_csv("mmr2_clean.csv", index=False)

latest = d[d["period"] == "2024/25"]
print(latest[["area","coverage","lci","uci","gap_to_95","gap_to_england","sig_below_95","sig_below_england"]])
first = d[d["period"] == "2010/11"].set_index("area")["coverage"]
last = d[d["period"] == "2024/25"].set_index("area")["coverage"]
print("Change 2010/11 to 2024/25 (percentage points):\n", (last - first).round(1))

          area  coverage   lci   uci  gap_to_95  gap_to_england  sig_below_95  \
14  Birmingham      75.5  74.9  76.2       19.5             8.2          True   
44     England      83.7   NaN   NaN       11.3             0.0         False   
29      Surrey      87.0  86.4  87.6        8.0            -3.3          True   

    sig_below_england  
14               True  
44              False  
29              False  
Change 2010/11 to 2024/25 (percentage points):
 area
Birmingham   -12.0
England       -0.5
Surrey        16.9
Name: coverage, dtype: float64
